# Notebook 02 — Train your own normal facial-skin LoRA (SDXL, Kaggle)

Train **one SDXL LoRA** using real normal-face images from the *frozen train partition only*. Captions can describe photograph framing, lighting, and unretouched skin texture but must not invent clinical details. The pretrained `profpeng/skinenhancer` LoRA is **not** part of this training operation: it is a separate optional inference comparison in Notebook 01.

First use the **20-step smoke test**, then choose longer training based on Kaggle memory and observed image quality. T4 GPU success is not guaranteed: SDXL is relatively demanding. The notebook does not train from scratch.

## 1. Installation and environment

In [ ]:
import subprocess,sys
PINNED = ['diffusers==0.35.1','transformers==4.56.2','accelerate==1.10.1',
          'peft==0.17.1','datasets==3.6.0','huggingface_hub==0.34.4',
          'safetensors==0.6.2','bitsandbytes==0.47.0','imagehash==4.3.2',
          'ftfy==6.3.1']
subprocess.check_call([sys.executable,'-m','pip','install','-q',*PINNED])
print('Installed pinned training dependencies; restart if Kaggle preloaded conflicting packages.')

In [ ]:
import os, json, csv, math, hashlib, shutil, subprocess, sys, re
from pathlib import Path
import numpy as np
import torch, diffusers, transformers, datasets, peft, accelerate
from PIL import Image, ImageOps
from IPython.display import display
assert torch.cuda.is_available(), 'Enable Kaggle T4 GPU in settings.'
print('GPU:', torch.cuda.get_device_name(0), 'torch:',torch.__version__, 'diffusers:',diffusers.__version__)

## 2. Kaggle v2 real-data source and approved train split
Only `/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection` is allowed as the original real image collection. The cell below explicitly points to it, but **the actual LoRA training images are staged only from the human-reviewed frozen real TRAIN manifest**. Attach `real_all_manifest.csv`, the saved deduplicated split images, and the passed label-audit JSON. The notebook refuses raw unsplit images, missing approvals and test/validation leakage. Training logs and weights remain in the Kaggle Output folder; do not commit private images or large checkpoints to GitHub automatically.


In [ ]:
IMAGE_ROOT = Path('/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection/Face_Skin_dataset/Normal_Skin')  # CHANGE THIS
SPLIT_CSV=''                      # optional CSV: filename,split
CAPTIONS_CSV=''                   # optional CSV: filename,caption
ALLOW_PROVISIONAL_SPLIT=False    # change only if no final global splits exist
BASE_MODEL='stabilityai/stable-diffusion-xl-base-1.0'
VAE_MODEL='madebyollin/sdxl-vae-fp16-fix'
DATA_PREP=Path('/kaggle/working/normal_skin_sdxl_training_data')
LORA_OUT=Path('/kaggle/working/normal_skin_sdxl_lora')
STEPS=20                         # FIRST RUN smoke test; increase after successful test
RESOLUTION=512                   # conservative starting point for T4
RANK=8
TRAIN_BATCH_SIZE=1
GRAD_ACCUM=4
LEARNING_RATE=1e-4
SEED=561231
CHECKPOINT_STEPS=100
assert RESOLUTION % 8 == 0 and STEPS>0
if not IMAGE_ROOT.exists():
    raise FileNotFoundError(f'Edit IMAGE_ROOT to the attached Kaggle dataset: {IMAGE_ROOT}')
if DATA_PREP.exists():
    raise FileExistsError(f'{DATA_PREP} already exists. Set a fresh directory per run to prevent stale images.')
if LORA_OUT.exists():
    raise FileExistsError(f'{LORA_OUT} already exists. Use a fresh output path for a reproducible run.')
DATA_PREP.mkdir(parents=True)
LORA_OUT.mkdir(parents=True)

In [ ]:
# SOURCE: sole Kaggle Facial Skin v2 collection; actual LoRA input is the APPROVED real train split ONLY.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prepare_lora_train_only,append_run_log
RAW_REAL_DATASET_V2='/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection'
REAL_TRAIN_MANIFEST='/kaggle/input/EDIT_APPROVED_REAL_SPLIT/real_all_manifest.csv'
REAL_TRAIN_IMAGE_ROOT='/kaggle/input/EDIT_APPROVED_REAL_IMAGES/facial_real_split_v2'
PASSED_REAL_LABEL_AUDIT='/kaggle/input/EDIT_APPROVED_REAL_AUDIT/label_audit_summary.json'
IMAGE_ROOT,real_train_evidence=prepare_lora_train_only(
    real_split_manifest=REAL_TRAIN_MANIFEST,real_image_root=REAL_TRAIN_IMAGE_ROOT,
    cls='Normal',stage_dir='/kaggle/working/approved_Normal_Skin_lora_input',
    origin_root=RAW_REAL_DATASET_V2,label_audit_json=PASSED_REAL_LABEL_AUDIT)
AUTO_SPLIT=False
ALLOW_PROVISIONAL_SPLIT=False
SPLIT_CSV=''
print('Approved real training images:',real_train_evidence['count'],'Folder:',IMAGE_ROOT)
# Capture full trainer stdout/stderr in an append-only research log while streaming to Kaggle.
def run_logged(command,path):
    import subprocess
    from pathlib import Path
    log=Path(path);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8',errors='replace') as f:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
        for line in proc.stdout:
            print(line,end='');f.write(line);f.flush()
        code=proc.wait()
    if code:raise subprocess.CalledProcessError(code,command)


## 3. Read real images and choose TRAIN only

Use an existing `train/` folder or explicit split CSV. If no globally frozen split exists, the provisional pHash split is **opt-in**, not equivalent to patient-level grouping. Captions map to filenames/relative paths without inferring diagnoses. **Explicit split leakage screen:** if train and withheld photos are near-duplicates (pHash distance <= 5), stop and review split assignments before training.

In [ ]:
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
import imagehash

EXTS={'.jpg','.jpeg','.png','.webp'}
paths=sorted(p for p in IMAGE_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
if not paths: raise FileNotFoundError('No images found under IMAGE_ROOT')
relpaths=[p.relative_to(IMAGE_ROOT).as_posix() for p in paths]
counts=Counter(p.name for p in paths)

def read_csv_mapping(path, field):
    if not path: return {}
    mapping={}
    with open(path,encoding='utf-8',newline='') as f:
        for row in csv.DictReader(f):
            if 'filename' not in row or field not in row:
                raise ValueError(f'CSV requires filename,{field} columns')
            mapping[row['filename'].replace('\\','/')] = row[field].strip()
    return mapping

splits=read_csv_mapping(SPLIT_CSV,'split')
captions=read_csv_mapping(CAPTIONS_CSV,'caption')
def lookup(mapping, path):
    relative=path.relative_to(IMAGE_ROOT).as_posix()
    if relative in mapping: return mapping[relative]
    if counts[path.name]==1: return mapping.get(path.name)
    return None

train_folder=IMAGE_ROOT/'train'
if SPLIT_CSV:
    selected=[p for p in paths if lookup(splits,p)=='train']
    print('Using explicit train split CSV')
elif train_folder.is_dir():
    selected=[p for p in paths if train_folder in p.parents]
    print('Using existing train/ folder')
elif ALLOW_PROVISIONAL_SPLIT:
    valid=[]; hashes=[]
    for p in paths:
        try:
            with Image.open(p) as im:
                im=ImageOps.exif_transpose(im).convert('RGB')
                hashes.append(imagehash.phash(im)); valid.append(p)
        except Exception as exc:
            print('Skipping unreadable image',p,exc)
    if len(valid)<4: raise ValueError('At least four decodable images needed for provisional split.')
    parent=list(range(len(valid)))
    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for i in range(len(hashes)):
        for j in range(i+1,len(hashes)):
            if hashes[i]-hashes[j] <=5:
                a,b=find(i),find(j)
                if a!=b: parent[b]=a
    groups=np.array([find(i) for i in range(len(valid))])
    if len(set(groups))<3: raise ValueError('Not enough independent image groups to split.')
    gss=GroupShuffleSplit(n_splits=1,test_size=0.30,random_state=SEED)
    idx_train,_=next(gss.split(np.arange(len(valid)),groups=groups))
    selected=[valid[i] for i in idx_train]
    print('Provisional pHash-group train split; NOT a patient-independent validation.')
else:
    raise ValueError('Provide train/ or SPLIT_CSV, or explicitly opt into ALLOW_PROVISIONAL_SPLIT.')
if not selected: raise ValueError('No images selected for training.')

# Leak check: explicit train/ or split CSV could contain a copy of a held-out image.
# This is a conservative near-duplicate flag (pHash <=5). It may raise on
# coincidentally similar portraits; review the identified pairs rather than
# bypassing the check or moving test images into training.
heldout = [p for p in paths if p not in set(selected)]
if heldout:
    def safe_hash(fp):
        try:
            with Image.open(fp) as raw:
                return imagehash.phash(ImageOps.exif_transpose(raw).convert('RGB'))
        except Exception as exc:
            print('Skipping unreadable image during split audit:', fp, exc)
            return None
    train_h = [(p, safe_hash(p)) for p in selected]
    held_h = [(p, safe_hash(p)) for p in heldout]
    conflicts = [(str(p), str(q), a-b) for p,a in train_h if a is not None
                 for q,b in held_h if b is not None and a-b <= 5]
    if conflicts:
        print('Near-duplicate train/held-out examples:', conflicts[:5])
        raise ValueError(f'Found {len(conflicts)} train/held-out pHash collisions (<=5). Review global grouped splits before training.')
    print('Held-out leakage screen passed:',len(selected),'train vs',len(heldout),'held-out images')
print('Training images selected:',len(selected))

## 4. Nonblack, aspect-ratio-preserving RGB preprocessing + captioned ImageFolder
Composite transparency on light neutral grey; EXIF-correct and letterbox instead of deforming rectangular portraits. The output contains only training JPEGs in `images/` and a root `metadata.jsonl` with `images/<file>.jpg` relative paths.

In [ ]:
from PIL import ImageEnhance
TARGET=RESOLUTION
folder=DATA_PREP/'images'; folder.mkdir(parents=True)
meta=DATA_PREP/'metadata.jsonl'
records=[]; skips=[]
for i,p in enumerate(selected):
    try:
        with Image.open(p) as raw:
            oriented=ImageOps.exif_transpose(raw)
            if oriented.mode in ('RGBA','LA') or 'transparency' in oriented.info:
                rgba=oriented.convert('RGBA')
                bg=Image.new('RGBA',rgba.size,(225,225,225,255)); bg.alpha_composite(rgba)
                im=bg.convert('RGB')
            else: im=oriented.convert('RGB')
            if min(im.size)<64: raise ValueError('Image too small')
            im.thumbnail((TARGET,TARGET),Image.Resampling.LANCZOS)
            canvas=Image.new('RGB',(TARGET,TARGET),(225,225,225))
            canvas.paste(im,((TARGET-im.width)//2,(TARGET-im.height)//2))
            name=f'normal_{i:06d}.jpg'; canvas.save(folder/name,quality=94,subsampling=0)
        caption=lookup(captions,p)
        if not caption: caption='Clinical documentary facial photograph, natural unretouched normal facial skin texture, realistic pores, neutral background'
        records.append({'file_name':'images/'+name,'text':caption.strip()})
    except Exception as exc:
        skips.append({'path':str(p),'error':str(exc)})
if not records: raise RuntimeError('All training images failed preprocessing.')
with meta.open('w',encoding='utf-8') as f:
    for r in records: f.write(json.dumps(r,ensure_ascii=False)+'\n')
(LORA_OUT/'excluded.json').write_text(json.dumps(skips,indent=2),encoding='utf-8')
print('Prepared',len(records),'images; skipped',len(skips))
# CPU sanity check: different corner pixels should not be black because of transparent input.
with Image.open(folder/Path(records[0]['file_name']).name) as check:
    print('Sample image',check.size,check.mode,'corner:',check.getpixel((0,0)))

## 5. Preview real training inputs

In [ ]:
from PIL import Image
previews=[]
for rec in records[:6]:
    with Image.open(DATA_PREP/rec['file_name']) as im: previews.append(im.copy())
if previews:
    sheet=Image.new('RGB',(RESOLUTION*min(3,len(previews)), RESOLUTION*math.ceil(len(previews)/3)),(255,255,255))
    for i,im in enumerate(previews): sheet.paste(im,((i%3)*RESOLUTION,(i//3)*RESOLUTION))
    display(sheet.resize((min(1000,sheet.width),int(sheet.height*min(1000,sheet.width)/sheet.width))))

## 6. Download **version-matched** official Diffusers SDXL LoRA trainer and check CLI
The script is pinned to Diffusers **v0.35.1**, not the moving `main` branch. Its `--help` output is checked before starting any training.

In [ ]:
from urllib.request import urlretrieve
SCRIPT=Path('/kaggle/working/train_text_to_image_lora_sdxl_v0351.py')
URL='https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/examples/text_to_image/train_text_to_image_lora_sdxl.py'
if not SCRIPT.exists():
    urlretrieve(URL, SCRIPT)
raw=SCRIPT.read_text(encoding='utf-8')
if 'check_min_version("0.35.0")' not in raw and "check_min_version('0.35.0')" not in raw:
    raise RuntimeError('Downloaded script does not appear to match the expected Diffusers v0.35.x training example')
help_result=subprocess.run([sys.executable,str(SCRIPT),'--help'], text=True,capture_output=True)
if help_result.returncode:
    raise RuntimeError('Official trainer did not import / run --help:\n'+help_result.stderr[-2500:])
for expected in ('--rank','--gradient_checkpointing','--use_8bit_adam','--max_train_steps','--mixed_precision'):
    assert expected in help_result.stdout, f'Official trainer missing expected flag {expected}'
print('Trainer CLI preflight passed; script SHA256:',hashlib.sha256(SCRIPT.read_bytes()).hexdigest())

## 7. Run the 20-step Kaggle T4 training smoke test

The official version-matched Diffusers v0.35.1 SDXL LoRA script is used with FP16, 8-bit Adam, gradient checkpointing and 512×512 padded inputs. `STEPS=20` is a smoke-test default, **not** a completed model. When running longer training use a fresh output directory, inspect sample images and monitor GPU memory. SDXL LoRA training on T4 can exceed 16 GB in some configurations; no local CUDA run is claimed.

In [ ]:
from huggingface_hub import model_info
base_sha=model_info(BASE_MODEL).sha
vae_sha=model_info(VAE_MODEL).sha
config={'base_model':BASE_MODEL,'base_commit':base_sha,'vae_model':VAE_MODEL,'vae_commit':vae_sha,
        'train_count':len(records),'resolution':RESOLUTION,'rank':RANK,'steps':STEPS,
        'optimizer':'AdamW8bit','mixed_precision':'fp16','seed':SEED,
        'training_script':'diffusers/v0.35.1/train_text_to_image_lora_sdxl.py'}
(LORA_OUT/'training_provenance.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
cmd=[sys.executable,'-m','accelerate.commands.launch','--num_processes=1','--mixed_precision=fp16',str(SCRIPT),
     '--pretrained_model_name_or_path', BASE_MODEL,
     '--pretrained_vae_model_name_or_path',VAE_MODEL,
     '--train_data_dir',str(DATA_PREP),
     '--image_column','image','--caption_column','text',
     '--resolution',str(RESOLUTION),'--train_batch_size',str(TRAIN_BATCH_SIZE),
     '--gradient_accumulation_steps',str(GRAD_ACCUM),
     '--max_train_steps',str(STEPS),'--learning_rate',str(LEARNING_RATE),
     '--lr_scheduler','constant','--lr_warmup_steps','0',
     '--checkpointing_steps',str(CHECKPOINT_STEPS),'--rank',str(RANK),
     '--output_dir',str(LORA_OUT),'--seed',str(SEED),
     '--mixed_precision','fp16','--gradient_checkpointing',
     '--use_8bit_adam','--report_to','none',
]
print('Training starts with',STEPS,'steps; input images:',len(records))
run_logged(cmd,LORA_OUT/'trainer_console.log')
weights=LORA_OUT/'pytorch_lora_weights.safetensors'
if not weights.is_file():
    raise FileNotFoundError(f'Expected trained LoRA file not found: {weights}')
print('LoRA weights:',weights,'size (MB):',round(weights.stat().st_size/1e6,2))

In [ ]:
# Save durable metadata, model provenance and checksum manifest for the Kaggle Output tab.
from dermsynreal.kaggle_runs import append_run_log
append_run_log(LORA_OUT,kind='lora_fine_tuning',config={'class':'Normal',
    'base_model':BASE_MODEL,'training_steps':STEPS,'seed':SEED,
    'approved_real_train_count':real_train_evidence['count'],
    'approved_manifest_sha256':real_train_evidence['manifest_sha256']},
    artifacts=[LORA_OUT/'trainer_console.log',LORA_OUT/'pytorch_lora_weights.safetensors'],
    repo_root=DERMSYNREAL_REPO)
print('LoRA output, logs and provenance:',LORA_OUT)

## 8. ZIP your trained LoRA and provenance (Kaggle Output)
After successful training, attach the exported `.safetensors` or entire LoRA folder to Notebook 01 and set `CUSTOM_LORA_SOURCE`. **An SD2.1 LoRA will not work with SDXL.**

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
zpath=Path('/kaggle/working/normal_face_sdxl_lora.zip')
with ZipFile(zpath,'w',compression=ZIP_DEFLATED) as z:
    for fp in (LORA_OUT/'pytorch_lora_weights.safetensors',LORA_OUT/'training_provenance.json'):
        assert fp.is_file(), str(fp)
        z.write(fp,arcname=fp.name)
print('Export:',zpath)